# Решения: stack/queue/deque

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('orders_slim.csv')
df = pd.read_csv(
    CSV_PATH,
    parse_dates=['order_purchase_timestamp', 'order_estimated_delivery_date', 'order_delivered_customer_date'],
)

from collections import deque

In [ ]:
events = df[['order_id', 'is_late']].head(10).copy()
events['action'] = np.where(events['is_late'] == 1, 'expedite', 'standard')
stack = []
for step in ['join_orders', 'drop_na', 'make_delay', 'group_state']:
    stack.append(step)
last_undo = stack.pop()
queue = deque(events['order_id'].tolist())
first_done = queue.popleft()
buffer = deque(maxlen=5)
for oid in events['order_id'].tolist():
    buffer.append(oid)
snapshot = list(buffer)
ops = ['join_orders', 'drop_na', 'make_delay', 'group_state', 'kmeans', 'report']
undo_order = list(reversed(ops))
late_ids = df.loc[df['is_late'] == 1, 'order_id'].head(3).tolist()
normal_ids = df.loc[df['is_late'] == 0, 'order_id'].head(3).tolist()
q = deque(normal_ids)
for oid in reversed(late_ids):
    q.appendleft(oid)
late_first = q[0]
STRUCT_NOTE = (
    'Stack подходит для undo preprocessing, потому что отменяем последнее действие. '
    'Queue и deque подходят для потока заказов: FIFO и быстрые операции с концов буфера.'
)
print(events.head())
print(last_undo, first_done, snapshot)
print(undo_order, late_first)
print(STRUCT_NOTE)